In [ ]:
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Using device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DERMNET_TRAIN_ROOT = "/content/dermnet_data"       # category folders directly inside
DERMNET_TEST_ROOT = "/content/dermnet_data_test"

def ensure_unzipped(zip_name, dest_dir):
    if os.path.isdir(dest_dir) and any(Path(dest_dir).iterdir()):
        print(f"{dest_dir} already populated — skipping unzip.")
        return
    print(f"Unzipping {zip_name} -> {dest_dir} ...")
    os.system(f"cp /content/drive/MyDrive/{zip_name} /content/{zip_name}")
    os.system(f"unzip -q /content/{zip_name} -d {dest_dir}")
    print("Done.")

ensure_unzipped("train.zip", DERMNET_TRAIN_ROOT)
ensure_unzipped("test.zip", DERMNET_TEST_ROOT)

Mounted at /content/drive
Unzipping train.zip -> /content/dermnet_data ...
Done.
Unzipping test.zip -> /content/dermnet_data_test ...
Done.


In [ ]:
CLASS_FOLDER_MAP = {
    "Eczema": "Eczema Photos",
    "Hives": "Urticaria Hives",
    "Acne/Rosacea": "Acne and Rosacea Photos",
    "Psoriasis/Lichen Planus": "Psoriasis pictures Lichen Planus and related diseases",
    "Contact Dermatitis/Poison Ivy": "Poison Ivy Photos and other Contact Dermatitis",
    "Ringworm/Fungal Infections": "Tinea Ringworm Candidiasis and other Fungal Infections",
}
CLASS_NAMES = list(CLASS_FOLDER_MAP.keys())
NUM_CLASSES = len(CLASS_NAMES)

In [ ]:
# DermNet has train/ and test/ only but no val/. Split train/ into
# train/val (85/15) and leave test/ completely untouched as
# the final evaluation later
VAL_FRACTION = 0.15

def collect_filepaths(root):
    samples = []
    root_dir = Path(root)
    for class_idx, class_name in enumerate(CLASS_NAMES):
        folder = root_dir / CLASS_FOLDER_MAP[class_name]
        if not folder.exists():
            raise FileNotFoundError(
                f"Expected folder not found: {folder}"
            )
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            samples.extend((str(p), class_idx) for p in folder.glob(ext))
    return samples


all_train_samples = collect_filepaths(DERMNET_TRAIN_ROOT)
test_samples = collect_filepaths(DERMNET_TEST_ROOT)

train_paths = [s[0] for s in all_train_samples]
train_labels = [s[1] for s in all_train_samples]

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels,
    test_size=VAL_FRACTION,
    stratify=train_labels,
    random_state=SEED,
)

print("Split sizes:")
print(f"  Train: {len(train_paths)}")
print(f"  Val:   {len(val_paths)}")
print(f"  Test:  {len(test_samples)}")

train_counts = Counter(train_labels)
for idx, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {train_counts[idx]}")

test_label_list = [s[1] for s in test_samples]
test_counts = Counter(test_label_list)
MIN_RELIABLE_TEST_COUNT = 15

thin_test_classes = []
for idx, name in enumerate(CLASS_NAMES):
    count = test_counts[idx]
    flag = ""
    if count < MIN_RELIABLE_TEST_COUNT:
        thin_test_classes.append(name)
    print(f"  {name}: {count}{flag}")

Split sizes:
  Train: 4464
  Val:   788
  Test:  1416  (untouched holdout, reserved for Day 6)

Per-class train counts (flag anything that looks thinner than expected):
  Eczema: 1050
  Hives: 180
  Acne/Rosacea: 714
  Psoriasis/Lichen Planus: 1194
  Contact Dermatitis/Poison Ivy: 221
  Ringworm/Fungal Infections: 1105

Per-class TEST counts (this is your Day 6 holdout — check now, not later):
  Eczema: 309
  Hives: 53
  Acne/Rosacea: 312
  Psoriasis/Lichen Planus: 352
  Contact Dermatitis/Poison Ivy: 65
  Ringworm/Fungal Infections: 325


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
class SkinConditionDataset(Dataset):
    def __init__(self, filepaths, labels, transform):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert("RGB")
        img = self.transform(img)
        return img, self.labels[idx]


test_paths = [s[0] for s in test_samples]
test_labels = [s[1] for s in test_samples]

train_ds = SkinConditionDataset(train_paths, train_labels, train_transform)
val_ds = SkinConditionDataset(val_paths, val_labels, eval_transform)
test_ds = SkinConditionDataset(test_paths, test_labels, eval_transform)  # not touched in this script

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
class_counts = np.array([train_counts[i] for i in range(NUM_CLASSES)])
class_weights = class_counts.sum() / (NUM_CLASSES * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print("Class weights (higher = rarer class, penalized more in the loss):")
for name, w in zip(CLASS_NAMES, class_weights.tolist()):
    print(f"  {name}: {w:.3f}")


Class weights (higher = rarer class, penalized more in the loss):
  Eczema: 0.709
  Hives: 4.133
  Acne/Rosacea: 1.042
  Psoriasis/Lichen Planus: 0.623
  Contact Dermatitis/Poison Ivy: 3.367
  Ringworm/Fungal Infections: 0.673


In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

for param in model.parameters():
    param.requires_grad = False

# replace classifier head only
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

trainable_params = [p for p in model.parameters() if p.requires_grad]
print(f"\nTrainable params: {sum(p.numel() for p in trainable_params):,} "
      f"(of {sum(p.numel() for p in model.parameters()):,} total)")


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 88.5MB/s]



Trainable params: 7,686 (of 4,015,234 total)


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(trainable_params, lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2
)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    return avg_loss, acc, precision, recall, f1, all_preds, all_labels

In [ ]:
NUM_EPOCHS = 15
CHECKPOINT_PATH = "/content/drive/MyDrive/skin_ai_app/best_model_stage1.pt" # meant for colab + drive syncing
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

FORCE_RETRAIN = True

if not FORCE_RETRAIN and os.path.exists(CHECKPOINT_PATH):
    print(f"Found existing checkpoint at {CHECKPOINT_PATH} — loading instead of retraining.")
    print("Set FORCE_RETRAIN = True above if you want a fresh run.")
    _existing = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(_existing["model_state_dict"])
    best_val_f1 = _existing["val_f1"]
    history = []
    print(f"Loaded checkpoint: epoch {_existing['epoch']}, val_macroF1={best_val_f1:.3f}")
else:
    best_val_f1 = 0.0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc, *_ = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_prec, val_rec, val_f1, _, _ = run_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)

        print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} val_macroF1={val_f1:.3f}")

        history.append({
            "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
            "val_loss": val_loss, "val_acc": val_acc,
            "val_precision": val_prec, "val_recall": val_rec, "val_f1": val_f1,
        })

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({
                "model_state_dict": model.state_dict(),
                "class_names": CLASS_NAMES,
                "epoch": epoch,
                "val_f1": val_f1,
            }, CHECKPOINT_PATH)
            print(f"  -> New best model saved (val_macroF1={val_f1:.3f})")

Epoch  1/15 | train_loss=1.5692 train_acc=0.389 | val_loss=1.4093 val_acc=0.466 val_macroF1=0.416
  -> New best model saved (val_macroF1=0.416)
Epoch  2/15 | train_loss=1.3568 train_acc=0.460 | val_loss=1.3438 val_acc=0.503 val_macroF1=0.462
  -> New best model saved (val_macroF1=0.462)
Epoch  3/15 | train_loss=1.2940 train_acc=0.481 | val_loss=1.3215 val_acc=0.528 val_macroF1=0.484
  -> New best model saved (val_macroF1=0.484)
Epoch  4/15 | train_loss=1.2487 train_acc=0.499 | val_loss=1.2775 val_acc=0.524 val_macroF1=0.484
  -> New best model saved (val_macroF1=0.484)
Epoch  5/15 | train_loss=1.2078 train_acc=0.507 | val_loss=1.2638 val_acc=0.536 val_macroF1=0.488
  -> New best model saved (val_macroF1=0.488)
Epoch  6/15 | train_loss=1.1815 train_acc=0.518 | val_loss=1.2595 val_acc=0.541 val_macroF1=0.499
  -> New best model saved (val_macroF1=0.499)
Epoch  7/15 | train_loss=1.1450 train_acc=0.537 | val_loss=1.2516 val_acc=0.552 val_macroF1=0.514
  -> New best model saved (val_macroF1

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH)
model.load_state_dict(checkpoint["model_state_dict"])

_, _, _, _, _, val_preds, val_labels_final = run_epoch(model, val_loader, criterion)
precision, recall, f1, support = precision_recall_fscore_support(
    val_labels_final, val_preds, average=None, zero_division=0
)

print(f"\nBest checkpoint: epoch {checkpoint['epoch']}, val_macroF1={checkpoint['val_f1']:.3f}\n")
print(f"{'Class':<35}{'Precision':>10}{'Recall':>10}{'F1':>10}{'Support':>10}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<35}{precision[i]:>10.3f}{recall[i]:>10.3f}{f1[i]:>10.3f}{support[i]:>10d}")


Best checkpoint: epoch 15, val_macroF1=0.535

Class                               Precision    Recall        F1   Support
Eczema                                  0.584     0.584     0.584       185
Hives                                   0.302     0.594     0.400        32
Acne/Rosacea                            0.683     0.754     0.717       126
Psoriasis/Lichen Planus                 0.599     0.431     0.501       211
Contact Dermatitis/Poison Ivy           0.379     0.564     0.454        39
Ringworm/Fungal Infections              0.560     0.549     0.554       195
